## tl;dr

**FINAL_BEST_DUALTRENDFUTURESV2R_RETAINED**. The final V3R challenger failed 1 absolute and 6 incumbent-superiority gates, so V2R remains the strongest tested research candidate and active credential-free forward dry-run.

- Validation: V2R 37.85% / PF 1.689 / DD 9.35%; V3R 38.56% / PF 1.693 / DD 10.70%.
- Current 2026: V2R 9.85% / PF 1.852; V3R 1.16% / PF 1.067.


## Context & Methods

This is the final locked tournament between `DualTrendFuturesV2R` and the
single Chandelier-exit challenger `DualTrendFuturesV3R`. The notebook reads raw
Freqtrade archives, evaluates annual folds and fee stress, runs a deterministic
four-week block bootstrap, applies every pre-declared gate, and verifies the
selected credential-free dry-run runtime.

### Key assumptions

- Historical windows were already visible and are not a sealed holdout.
- The fixed current universe creates survivorship bias.
- Weekly block bootstrap is a resampling stress test, not a market forecast.
- Dry-run and historical results cannot guarantee future profit.


## Data

### 1. Load the tournament manifest and raw archives

In [1]:
from pathlib import Path
import base64
import io
import json
import urllib.request
import zipfile
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
BACKTEST_DIR = PROJECT_ROOT / "user_data" / "backtest_results"
REPORT_DIR = PROJECT_ROOT / "user_data" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
manifest = json.loads((REPORT_DIR / "final_tournament_manifest.json").read_text(encoding="utf-8"))
STRATEGIES = tuple(manifest["strategies"])

records, raw, wallets = [], {}, {}
for run in manifest["runs"]:
    zip_path = BACKTEST_DIR / run["artifact"]
    base = zip_path.name.removesuffix(".zip")
    with zipfile.ZipFile(zip_path) as archive:
        result = json.loads(archive.read(f"{base}.json"))
        for strategy in STRATEGIES:
            stats = result["strategy"][strategy]
            wallet_member = f"{base}_{strategy}_wallet.feather"
            wallet = pd.read_feather(io.BytesIO(archive.read(wallet_member)))
            records.append({
                "artifact": zip_path.name, "strategy": strategy, "split": run["split"],
                "fee_per_side": run["fee_per_side"], "trades": int(stats["total_trades"]),
                "long_trades": int(stats["trade_count_long"]), "short_trades": int(stats["trade_count_short"]),
                "return_pct": 100 * float(stats["profit_total"]),
                "profit_abs_usdt": float(stats["profit_total_abs"]),
                "profit_factor": float(stats["profit_factor"]),
                "max_drawdown_pct": 100 * float(stats["max_drawdown_account"]),
                "market_change_pct": 100 * float(stats["market_change"]),
            })
            raw[(strategy, run["split"])] = stats
            wallets[(strategy, run["split"])] = wallet

summary = pd.DataFrame(records).sort_values(["split", "strategy"]).reset_index(drop=True)
required = {"backcast", "train", "validation", "2021", "2022", "2023", "2024", "2025", "current_2026", "fee_train", "fee_validation"}
assert set(summary.strategy) == set(STRATEGIES)
assert all(set(summary.loc[summary.strategy == s, "split"]) == required for s in STRATEGIES)
assert len(summary) == 22
summary.to_csv(REPORT_DIR / "final_tournament_summary.csv", index=False)
display(summary)

,artifact,strategy,split,fee_per_side,trades,long_trades,short_trades,return_pct,profit_abs_usdt,profit_factor,max_drawdown_pct,market_change_pct
0,backtest-result-2026-08-03_21-23-28.zip,DualTrendFuturesV2R,2021,0.0005,71,65,6,92.336881,9233.688139,2.485934,8.660666,991.602725
1,backtest-result-2026-08-03_21-23-28.zip,DualTrendFuturesV3R,2021,0.0005,62,57,5,106.592635,10659.263463,2.628171,10.846129,991.602725
2,backtest-result-2026-08-03_21-23-36.zip,DualTrendFuturesV2R,2022,0.0005,77,0,77,2.088624,208.862432,1.062774,14.407058,-64.304372
3,backtest-result-2026-08-03_21-23-36.zip,DualTrendFuturesV3R,2022,0.0005,80,0,80,1.483813,148.381257,1.041712,14.550144,-64.304372
4,backtest-result-2026-08-03_21-23-43.zip,DualTrendFuturesV2R,2023,0.0005,45,27,18,7.328304,732.830431,1.540460,5.523688,87.505267
5,backtest-result-2026-08-03_21-23-43.zip,DualTrendFuturesV3R,2023,0.0005,41,26,15,12.957931,1295.793094,2.237386,3.621049,87.505267
6,backtest-result-2026-08-03_21-23-51.zip,DualTrendFuturesV2R,2024,0.0005,65,43,22,28.261461,2826.146074,2.059263,8.739788,140.140040
7,backtest-result-2026-08-03_21-23-51.zip,DualTrendFuturesV3R,2024,0.0005,66,42,24,31.792481,3179.248139,2.255732,9.104669,140.140040
8,backtest-result-2026-08-03_21-24-00.zip,DualTrendFuturesV2R,2025,0.0005,63,37,26,7.517262,751.726229,1.342055,9.347932,-21.854841
9,backtest-result-2026-08-03_21-24-00.zip,DualTrendFuturesV3R,2025,0.0005,64,37,27,5.054662,505.466198,1.219348,9.982289,-21.854841


### 2. Compare incumbent and challenger

In [2]:
incumbent = summary[summary.strategy == "DualTrendFuturesV2R"].set_index("split")
challenger = summary[summary.strategy == "DualTrendFuturesV3R"].set_index("split")
ordered_splits = ["backcast", "train", "2021", "2022", "2023", "validation", "2024", "2025", "current_2026", "fee_train", "fee_validation"]
comparison_rows = []
for split in ordered_splits:
    i, c = incumbent.loc[split], challenger.loc[split]
    comparison_rows.append({
        "split": split, "v2r_trades": int(i.trades), "v3r_trades": int(c.trades),
        "v2r_return_pct": i.return_pct, "v3r_return_pct": c.return_pct,
        "return_delta_pp": c.return_pct - i.return_pct,
        "v2r_profit_factor": i.profit_factor, "v3r_profit_factor": c.profit_factor,
        "pf_delta": c.profit_factor - i.profit_factor,
        "v2r_max_drawdown_pct": i.max_drawdown_pct, "v3r_max_drawdown_pct": c.max_drawdown_pct,
        "drawdown_delta_pp": c.max_drawdown_pct - i.max_drawdown_pct,
        "v2r_return_to_dd": i.return_pct / i.max_drawdown_pct,
        "v3r_return_to_dd": c.return_pct / c.max_drawdown_pct,
        "return_to_dd_delta_pct": 100 * ((c.return_pct / c.max_drawdown_pct) / (i.return_pct / i.max_drawdown_pct) - 1),
    })
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(REPORT_DIR / "final_tournament_comparison.csv", index=False)
display(comparison)

,split,v2r_trades,v3r_trades,v2r_return_pct,v3r_return_pct,return_delta_pp,v2r_profit_factor,v3r_profit_factor,pf_delta,v2r_max_drawdown_pct,v3r_max_drawdown_pct,drawdown_delta_pp,v2r_return_to_dd,v3r_return_to_dd,return_to_dd_delta_pct
0,backcast,27,22,12.852366,15.933884,3.081518,2.266504,2.497091,0.230586,5.568551,7.709025,2.140474,2.308027,2.066913,-10.446754
1,train,192,183,111.011174,135.483947,24.472773,1.720009,1.839759,0.119750,17.077879,16.095232,-0.982647,6.500291,8.417645,29.496441
2,2021,71,62,92.336881,106.592635,14.255753,2.485934,2.628171,0.142237,8.660666,10.846129,2.185464,10.661638,9.827712,-7.821743
3,2022,77,80,2.088624,1.483813,-0.604812,1.062774,1.041712,-0.021062,14.407058,14.550144,0.143086,0.144972,0.101979,-29.656053
4,2023,45,41,7.328304,12.957931,5.629627,1.540460,2.237386,0.696925,5.523688,3.621049,-1.902639,1.326705,3.578502,169.728547
5,validation,128,130,37.846774,38.559121,0.712347,1.689225,1.692984,0.003759,9.352103,10.702199,1.350096,4.046873,3.602916,-10.970382
6,2024,65,66,28.261461,31.792481,3.531021,2.059263,2.255732,0.196469,8.739788,9.104669,0.364881,3.233655,3.491888,7.985781
7,2025,63,64,7.517262,5.054662,-2.462600,1.342055,1.219348,-0.122707,9.347932,9.982289,0.634357,0.804163,0.506363,-37.032301
8,current_2026,34,40,9.847238,1.156968,-8.690270,1.851733,1.067204,-0.784529,4.606070,10.524242,5.918172,2.137883,0.109934,-94.857828
9,fee_train,192,183,99.785886,123.604257,23.818371,1.634399,1.750764,0.116365,18.784250,17.296117,-1.488133,5.312210,7.146359,34.527028


### 3. Revalidate data quality and raw calculations

In [3]:
quality = pd.read_csv(REPORT_DIR / "dual_trend_data_quality.csv")
timeframe_reconciliation = pd.read_csv(REPORT_DIR / "dual_trend_timeframe_reconciliation.csv")
coverage_cols = [c for c in quality.columns if c.endswith("coverage_pct")]
assert quality[["duplicate_timestamps", "null_cells", "invalid_ohlcv"]].sum().sum() == 0
assert quality.largest_gap_candles.max() == 0
assert quality[coverage_cols].min().min() == 100.0
assert timeframe_reconciliation.mismatch_pct.max() <= 0.20

def recomputed_pf(values):
    gains = values[values > 0].sum()
    losses = -values[values < 0].sum()
    return float(gains / losses) if losses > 0 else float("inf")

reconciliation_rows = []
for (strategy, split), stats in raw.items():
    trades = pd.DataFrame(stats["trades"])
    reconciliation_rows.append({
        "strategy": strategy, "split": split,
        "stored_trades": int(stats["total_trades"]), "recomputed_trades": len(trades),
        "stored_profit_abs": float(stats["profit_total_abs"]), "recomputed_profit_abs": float(trades.profit_abs.sum()),
        "stored_pf": float(stats["profit_factor"]), "recomputed_pf": recomputed_pf(trades.profit_abs),
        "stored_wins": int(stats["wins"]), "recomputed_wins": int((trades.profit_abs > 0).sum()),
        "stored_losses": int(stats["losses"]), "recomputed_losses": int((trades.profit_abs < 0).sum()),
    })
reconciliation = pd.DataFrame(reconciliation_rows).sort_values(["strategy", "split"])
assert (reconciliation.stored_trades == reconciliation.recomputed_trades).all()
assert np.allclose(reconciliation.stored_profit_abs, reconciliation.recomputed_profit_abs, atol=1e-6)
assert np.allclose(reconciliation.stored_pf, reconciliation.recomputed_pf, atol=1e-12)
assert (reconciliation.stored_wins == reconciliation.recomputed_wins).all()
assert (reconciliation.stored_losses == reconciliation.recomputed_losses).all()
reconciliation.to_csv(REPORT_DIR / "final_tournament_reconciliation.csv", index=False)
display(reconciliation)

,strategy,split,stored_trades,recomputed_trades,stored_profit_abs,recomputed_profit_abs,stored_pf,recomputed_pf,stored_wins,recomputed_wins,stored_losses,recomputed_losses
4,DualTrendFuturesV2R,2021,71,71,9233.688139,9233.688139,2.485934,2.485934,25,25,46,46
6,DualTrendFuturesV2R,2022,77,77,208.862432,208.862432,1.062774,1.062774,26,26,51,51
8,DualTrendFuturesV2R,2023,45,45,732.830431,732.830431,1.540460,1.540460,17,17,28,28
10,DualTrendFuturesV2R,2024,65,65,2826.146074,2826.146074,2.059263,2.059263,27,27,38,38
12,DualTrendFuturesV2R,2025,63,63,751.726229,751.726229,1.342055,1.342055,29,29,34,34
0,DualTrendFuturesV2R,backcast,27,27,1285.236564,1285.236564,2.266504,2.266504,13,13,14,14
14,DualTrendFuturesV2R,current_2026,34,34,984.723849,984.723849,1.851733,1.851733,13,13,21,21
18,DualTrendFuturesV2R,fee_train,192,192,9978.588573,9978.588573,1.634399,1.634399,63,63,129,129
20,DualTrendFuturesV2R,fee_validation,128,128,3282.901992,3282.901992,1.582454,1.582454,54,54,74,74
2,DualTrendFuturesV2R,train,192,192,11101.117387,11101.117387,1.720009,1.720009,66,66,126,126


## Results

### 4. Run the deterministic weekly block bootstrap

In [4]:
def weekly_returns(strategy, split):
    wallet = wallets[(strategy, split)].copy()
    wallet["date"] = pd.to_datetime(wallet.date, utc=True)
    weekly = wallet.set_index("date").total_quote.resample("W-SUN").last().dropna().pct_change().dropna()
    return weekly.to_numpy(dtype=float)

def block_bootstrap(returns, seed, simulations=5000, block_size=4):
    rng = np.random.default_rng(seed)
    n = len(returns)
    blocks_needed = int(np.ceil(n / block_size))
    starts = rng.integers(0, n, size=(simulations, blocks_needed))
    offsets = np.arange(block_size)
    indices = (starts[:, :, None] + offsets[None, None, :]) % n
    sampled = returns[indices].reshape(simulations, -1)[:, :n]
    equity = np.cumprod(1.0 + sampled, axis=1)
    path = np.concatenate([np.ones((simulations, 1)), equity], axis=1)
    running_high = np.maximum.accumulate(path, axis=1)
    max_dd = 100 * np.max(1.0 - path / running_high, axis=1)
    terminal = 100 * (equity[:, -1] - 1.0)
    return terminal, max_dd

bootstrap_rows = []
for idx, strategy in enumerate(STRATEGIES):
    observed = np.concatenate([weekly_returns(strategy, "train"), weekly_returns(strategy, "validation")])
    terminal, max_dd = block_bootstrap(observed, seed=20260804 + idx)
    bootstrap_rows.append({
        "strategy": strategy, "observed_weeks": len(observed), "simulations": len(terminal), "block_weeks": 4,
        "terminal_return_p05_pct": float(np.quantile(terminal, 0.05)),
        "terminal_return_median_pct": float(np.quantile(terminal, 0.50)),
        "terminal_return_p95_pct": float(np.quantile(terminal, 0.95)),
        "max_drawdown_p50_pct": float(np.quantile(max_dd, 0.50)),
        "max_drawdown_p95_pct": float(np.quantile(max_dd, 0.95)),
    })
bootstrap = pd.DataFrame(bootstrap_rows).set_index("strategy")
bootstrap.to_csv(REPORT_DIR / "final_tournament_block_bootstrap.csv")
display(bootstrap)

,observed_weeks,simulations,block_weeks,terminal_return_p05_pct,terminal_return_median_pct,terminal_return_p95_pct,max_drawdown_p50_pct,max_drawdown_p95_pct
strategy,,,,,,,,
DualTrendFuturesV2R,261,5000,4,24.620790,174.754254,656.619068,14.756577,25.961717
DualTrendFuturesV3R,261,5000,4,21.528802,205.252243,905.615446,15.585463,27.789382


### 5. Apply absolute and incumbent-superiority gates

In [5]:
folds = ["2021", "2022", "2023", "2024", "2025", "current_2026"]
v3_fold_returns = challenger.loc[folds, "return_pct"]
v2_fold_returns = incumbent.loc[folds, "return_pct"]
v3_positive_folds = int((v3_fold_returns > 0).sum())
v2_positive_folds = int((v2_fold_returns > 0).sum())

val_stats = raw[("DualTrendFuturesV3R", "validation")]
pair_rows = pd.DataFrame([r for r in val_stats["results_per_pair"] if r["key"] != "TOTAL"])
val_trades = pd.DataFrame(val_stats["trades"])
positive_pairs = pair_rows[pair_rows.profit_total_abs > 0]
pair_concentration = float(positive_pairs.profit_total_abs.max() / positive_pairs.profit_total_abs.sum())
positive_trades = val_trades[val_trades.profit_abs > 0]
trade_concentration = float(positive_trades.profit_abs.max() / positive_trades.profit_abs.sum())

lookahead = pd.read_csv(REPORT_DIR / "DualTrendFuturesV3R_lookahead.csv").iloc[0]
technical = json.loads((REPORT_DIR / "final_v3r_technical_checks.json").read_text(encoding="utf-8"))
technical_pass = (str(lookahead.has_bias).lower() == "false" and int(lookahead.total_signals) == 100
                  and int(lookahead.biased_entry_signals) == 0 and int(lookahead.biased_exit_signals) == 0
                  and technical["recursive_analysis"]["status"] == "PASS"
                  and not technical["recursive_analysis"]["indicator_lookahead_found"]
                  and abs(float(technical["recursive_analysis"]["ema200_1d_variance_pct_at_800"])) <= 0.01)

absolute_gates = {
    "train": challenger.loc["train", "return_pct"] > 0 and challenger.loc["train", "profit_factor"] >= 1.20 and challenger.loc["train", "max_drawdown_pct"] <= 30,
    "validation": challenger.loc["validation", "trades"] >= 100 and challenger.loc["validation", "return_pct"] > 0 and challenger.loc["validation", "profit_factor"] >= 1.50 and challenger.loc["validation", "max_drawdown_pct"] <= 12,
    "five_positive_folds": v3_positive_folds >= 5,
    "worst_fold_above_minus_5": float(v3_fold_returns.min()) > -5,
    "median_fold_pf_1_20": float(challenger.loc[folds, "profit_factor"].median()) >= 1.20,
    "year_2025": challenger.loc["2025", "return_pct"] > 0 and challenger.loc["2025", "profit_factor"] >= 1.15 and challenger.loc["2025", "max_drawdown_pct"] <= 12 and challenger.loc["2025", "trades"] >= 20,
    "current_2026": challenger.loc["current_2026", "return_pct"] > 0 and challenger.loc["current_2026", "profit_factor"] >= 1.30 and challenger.loc["current_2026", "max_drawdown_pct"] <= 10 and challenger.loc["current_2026", "trades"] >= 12,
    "fee_validation": challenger.loc["fee_validation", "return_pct"] > 0 and challenger.loc["fee_validation", "profit_factor"] >= 1.30 and challenger.loc["fee_validation", "max_drawdown_pct"] <= 13,
    "fee_train": challenger.loc["fee_train", "return_pct"] > 0 and challenger.loc["fee_train", "profit_factor"] >= 1.20,
    "backcast": challenger.loc["backcast", "return_pct"] > 0 and challenger.loc["backcast", "profit_factor"] >= 1.0 and challenger.loc["backcast", "max_drawdown_pct"] <= 15 and challenger.loc["backcast", "trades"] >= 10,
    "five_positive_pairs": int((pair_rows.profit_total > 0).sum()) >= 5,
    "pair_concentration": pair_concentration <= 0.50,
    "trade_concentration": trade_concentration <= 0.25,
    "technical": technical_pass,
    "bootstrap_return_p05_positive": bootstrap.loc["DualTrendFuturesV3R", "terminal_return_p05_pct"] > 0,
    "bootstrap_dd_p95_35": bootstrap.loc["DualTrendFuturesV3R", "max_drawdown_p95_pct"] <= 35,
}

v2_val_ratio = incumbent.loc["validation", "return_pct"] / incumbent.loc["validation", "max_drawdown_pct"]
v3_val_ratio = challenger.loc["validation", "return_pct"] / challenger.loc["validation", "max_drawdown_pct"]
v2_fee_ratio = incumbent.loc["fee_validation", "return_pct"] / incumbent.loc["fee_validation", "max_drawdown_pct"]
v3_fee_ratio = challenger.loc["fee_validation", "return_pct"] / challenger.loc["fee_validation", "max_drawdown_pct"]
v2_train_ratio = incumbent.loc["train", "return_pct"] / incumbent.loc["train", "max_drawdown_pct"]
v3_train_ratio = challenger.loc["train", "return_pct"] / challenger.loc["train", "max_drawdown_pct"]

superiority_gates = {
    "validation_return": challenger.loc["validation", "return_pct"] >= incumbent.loc["validation", "return_pct"],
    "validation_pf": challenger.loc["validation", "profit_factor"] >= incumbent.loc["validation", "profit_factor"],
    "validation_return_dd_10pct": v3_val_ratio >= 1.10 * v2_val_ratio,
    "year_2025_return": challenger.loc["2025", "return_pct"] >= incumbent.loc["2025", "return_pct"],
    "current_2026_return": challenger.loc["current_2026", "return_pct"] >= incumbent.loc["current_2026", "return_pct"],
    "fee_validation_return_dd": v3_fee_ratio >= v2_fee_ratio,
    "train_return_dd_floor": v3_train_ratio >= 0.90 * v2_train_ratio,
    "positive_fold_count": v3_positive_folds >= v2_positive_folds,
    "worst_fold_return": float(v3_fold_returns.min()) >= float(v2_fold_returns.min()),
    "bootstrap_p05": bootstrap.loc["DualTrendFuturesV3R", "terminal_return_p05_pct"] >= bootstrap.loc["DualTrendFuturesV2R", "terminal_return_p05_pct"],
}
absolute_df = pd.DataFrame([{"gate": k, "pass": bool(v)} for k, v in absolute_gates.items()])
superiority_df = pd.DataFrame([{"gate": k, "pass": bool(v)} for k, v in superiority_gates.items()])
absolute_df.to_csv(REPORT_DIR / "final_v3r_absolute_gates.csv", index=False)
superiority_df.to_csv(REPORT_DIR / "final_v3r_superiority_gates.csv", index=False)
pair_rows[["key", "trades", "profit_total_pct", "profit_factor"]].to_csv(REPORT_DIR / "final_v3r_validation_pairs.csv", index=False)
display(absolute_df)
display(superiority_df)

,gate,pass
0,train,True
1,validation,True
2,five_positive_folds,True
3,worst_fold_above_minus_5,True
4,median_fold_pf_1_20,True
5,year_2025,True
6,current_2026,False
7,fee_validation,True
8,fee_train,True
9,backcast,True


,gate,pass
0,validation_return,True
1,validation_pf,True
2,validation_return_dd_10pct,False
3,year_2025_return,False
4,current_2026_return,False
5,fee_validation_return_dd,False
6,train_return_dd_floor,True
7,positive_fold_count,True
8,worst_fold_return,False
9,bootstrap_p05,False


### 6. Select the final strategy and verify runtime

In [6]:
challenger_wins = bool(absolute_df["pass"].all() and superiority_df["pass"].all())
selected_strategy = "DualTrendFuturesV3R" if challenger_wins else "DualTrendFuturesV2R"

base_config = json.loads((PROJECT_ROOT / "user_data" / "config.json").read_text(encoding="utf-8"))
v2r_config = json.loads((PROJECT_ROOT / "user_data" / "config-dual-futures-v2r-dry-run.json").read_text(encoding="utf-8"))
assert base_config["exchange"]["key"] == base_config["exchange"]["secret"] == ""
assert v2r_config["exchange"]["key"] == v2r_config["exchange"]["secret"] == ""
credentials = f'{base_config["api_server"]["username"]}:{base_config["api_server"]["password"]}'

def runtime_at(port):
    request = urllib.request.Request(f"http://127.0.0.1:{port}/api/v1/show_config")
    request.add_header("Authorization", "Basic " + base64.b64encode(credentials.encode()).decode())
    with urllib.request.urlopen(request, timeout=5) as response:
        cfg = json.loads(response.read())
    return {"http_status": int(response.status), "state": cfg["state"], "dry_run": bool(cfg["dry_run"]),
            "strategy": cfg["strategy"], "trading_mode": cfg["trading_mode"], "short_allowed": bool(cfg["short_allowed"]),
            "stake_amount": float(cfg["stake_amount"]), "max_open_trades": int(cfg["max_open_trades"])}

runtime_v1 = runtime_at(8080)
runtime_best = runtime_at(8081)
assert runtime_v1["strategy"] == "DualTrendFuturesV1" and runtime_v1["dry_run"] and runtime_v1["state"] == "running"
assert selected_strategy == "DualTrendFuturesV2R"
assert runtime_best["strategy"] == selected_strategy and runtime_best["dry_run"] and runtime_best["state"] == "running"
assert runtime_best["trading_mode"] == "futures" and runtime_best["short_allowed"] and runtime_best["stake_amount"] == 25 and runtime_best["max_open_trades"] == 2

failed_absolute = absolute_df.loc[~absolute_df["pass"], "gate"].tolist()
failed_superiority = superiority_df.loc[~superiority_df["pass"], "gate"].tolist()
decision = {
    "decision": "FINAL_BEST_DUALTRENDFUTURESV2R_RETAINED",
    "selected_strategy": selected_strategy, "rejected_challenger": "DualTrendFuturesV3R",
    "challenger_absolute_gates_pass": bool(absolute_df["pass"].all()),
    "challenger_superiority_gates_pass": bool(superiority_df["pass"].all()),
    "failed_absolute_gates": failed_absolute, "failed_superiority_gates": failed_superiority,
    "selected_runtime_verified": True, "selected_runtime_port": 8081,
    "primary_v1_runtime_verified": True, "primary_v1_port": 8080,
    "live_trading_authorized": False, "live_api_keys_present": False,
    "new_sealed_holdout": False, "future_profit_guaranteed": False,
}
(REPORT_DIR / "final_strategy_decision.json").write_text(json.dumps(decision, indent=2), encoding="utf-8")
(REPORT_DIR / "final_strategy_runtime.json").write_text(json.dumps({"primary": runtime_v1, "selected": runtime_best}, indent=2), encoding="utf-8")
decision

{'decision': 'FINAL_BEST_DUALTRENDFUTURESV2R_RETAINED',
 'selected_strategy': 'DualTrendFuturesV2R',
 'rejected_challenger': 'DualTrendFuturesV3R',
 'challenger_absolute_gates_pass': False,
 'challenger_superiority_gates_pass': False,
 'failed_absolute_gates': ['current_2026'],
 'failed_superiority_gates': ['validation_return_dd_10pct',
  'year_2025_return',
  'current_2026_return',
  'fee_validation_return_dd',
  'worst_fold_return',
  'bootstrap_p05'],
 'selected_runtime_verified': True,
 'selected_runtime_port': 8081,
 'primary_v1_runtime_verified': True,
 'primary_v1_port': 8080,
 'live_trading_authorized': False,
 'live_api_keys_present': False,
 'new_sealed_holdout': False,
 'future_profit_guaranteed': False}

## Takeaways

- Chandelier exits improved older train, 2021, 2023, and 2024 results, but materially weakened 2025 and the current bearish 2026 regime.
- V3R's validation return was only slightly higher while its drawdown was higher, so risk-adjusted validation performance was worse than V2R.
- The challenger passed technical bias checks but failed the pre-declared current-regime and superiority requirements; thresholds were not relaxed.
- V2R remains running on port 8081 and V1 remains the primary control on port 8080. V3R was never deployed.
- No historical or resampled result makes the strategy ideal or guarantees future profit; the active V2R dry-run is collecting the only new forward evidence.
